In [ ]:
from architectures_28x28.KKAN import * 

import torch
from torch import fx

In [ ]:
model = KKAN_Small()

In [ ]:
import torch
from torch import nn
 
example = (torch.randn(1, 3, 28, 28),)  

gm = torch.export.export(model, example)   # PT2E-Graph
print(gm.graph)                            # 1) rohe Textdarstellung

exported_model = gm.module()


graph():
    %p_conv1_convs_0_conv_base_weight : [num_users=1] = placeholder[target=p_conv1_convs_0_conv_base_weight]
    %p_conv1_convs_0_conv_spline_weight : [num_users=1] = placeholder[target=p_conv1_convs_0_conv_spline_weight]
    %p_conv1_convs_0_conv_spline_scaler : [num_users=1] = placeholder[target=p_conv1_convs_0_conv_spline_scaler]
    %p_conv1_convs_1_conv_base_weight : [num_users=1] = placeholder[target=p_conv1_convs_1_conv_base_weight]
    %p_conv1_convs_1_conv_spline_weight : [num_users=1] = placeholder[target=p_conv1_convs_1_conv_spline_weight]
    %p_conv1_convs_1_conv_spline_scaler : [num_users=1] = placeholder[target=p_conv1_convs_1_conv_spline_scaler]
    %p_conv1_convs_2_conv_base_weight : [num_users=1] = placeholder[target=p_conv1_convs_2_conv_base_weight]
    %p_conv1_convs_2_conv_spline_weight : [num_users=1] = placeholder[target=p_conv1_convs_2_conv_spline_weight]
    %p_conv1_convs_2_conv_spline_scaler : [num_users=1] = placeholder[target=p_conv1_convs_2_conv_s

In [ ]:
# 2) gezielt alle Nodes listen
for n in gm.graph.nodes:
    print(f"{n.op:12} {n.target}")

In [ ]:
# === 0) Imports & Setup =======================================================
import torch
from torch import nn

# Dein Modell (ggf. anpassen)
from architectures_28x28.KKAN import KKAN_Small

# Neu: PT2E Kernfunktionen (keine Deprecated-Warnungen mehr)
from torchao.quantization.pt2e.quantize_pt2e import (
    prepare_pt2e,
    convert_pt2e
)

from executorch.backends.xnnpack.quantizer.xnnpack_quantizer import (
  get_symmetric_quantization_config,
  XNNPACKQuantizer,
)

# === 1) Modell & Beispielinput =================================================
model = KKAN_Small().eval()
example = (torch.randn(1, 3, 28, 28),)

exported_model = torch.export.export(model, example).module()

quantizer = XNNPACKQuantizer()
quantizer.set_global(get_symmetric_quantization_config())

prepared_model = prepare_pt2e(exported_model, quantizer)
print(prepared_model.graph)

quantized_model = convert_pt2e(prepared_model)
print(quantized_model)

graph():
    %conv1_convs_0_conv_base_weight : [num_users=1] = get_attr[target=conv1.convs.0.conv.base_weight]
    %activation_post_process_1 : [num_users=1] = call_module[target=activation_post_process_1](args = (%conv1_convs_0_conv_base_weight,), kwargs = {})
    %conv1_convs_0_conv_spline_weight : [num_users=1] = get_attr[target=conv1.convs.0.conv.spline_weight]
    %activation_post_process_24 : [num_users=1] = call_module[target=activation_post_process_24](args = (%conv1_convs_0_conv_spline_weight,), kwargs = {})
    %conv1_convs_0_conv_spline_scaler : [num_users=1] = get_attr[target=conv1.convs.0.conv.spline_scaler]
    %conv1_convs_1_conv_base_weight : [num_users=1] = get_attr[target=conv1.convs.1.conv.base_weight]
    %activation_post_process_35 : [num_users=1] = call_module[target=activation_post_process_35](args = (%conv1_convs_1_conv_base_weight,), kwargs = {})
    %conv1_convs_1_conv_spline_weight : [num_users=1] = get_attr[target=conv1.convs.1.conv.spline_weight]
    %activ